# Этап 2. Система ретрива (извлечения)
## Банковский консультант: RAG-система

1. Базовый ретривер (Similarity Search + MMR)
2. Продвинутые техники (гибридный поиск BM25, фильтрация по метаданным, контекстное сжатие)
3. Оценка качества (Hit Rate@k, MRR)

## 0. Инициализация — подключение к pgvector

In [ ]:
import json
import time
import numpy as np
from dotenv import load_dotenv

load_dotenv()

# Загружаем конфиг (этап 1)
with open("rag_config.json", encoding="utf-8") as f:
    config = json.load(f)

CONNECTION_STRING = config["connection_string"]
COLLECTION_NAME   = config["collection_name"]

print("Конфигурация загружена:")
print(json.dumps(config, ensure_ascii=False, indent=2))

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import PGVector

# Загружаем модель эмбеддингов
embeddings_model = HuggingFaceEmbeddings(
    model_name=config["embedding_model"],
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True, "batch_size": 32},
)

# Подключаемся к существующей коллекции
vectorstore = PGVector(
    collection_name=COLLECTION_NAME,
    connection_string=CONNECTION_STRING,
    embedding_function=embeddings_model,
)

print(f"Подключено к pgvector, коллекция: {COLLECTION_NAME}")

---
## 1. Базовый ретривер
### 1.1 Similarity Search (поиск по косинусному сходству)

In [ ]:
def similarity_search(query: str, k: int = 4) -> list:
    """Поиск k наиболее похожих чанков по косинусному сходству."""
    results = vectorstore.similarity_search_with_score(query, k=k)
    return results


def print_results(results: list, title: str = "Результаты поиска"):
    """вывод результатов поиска."""
    print(f"  {title}")
    for i, (doc, score) in enumerate(results, 1):
        score_str = f"{score:.4f}" if score is not None else "N/A"
        print(f"\n[{i}] score={score_str} | {doc.metadata['title']} | тип: {doc.metadata['product_type']}")
        print(f"    {doc.page_content[:220].strip()}...")


# Тестовые запросы
test_queries = [
    "Какая минимальная сумма потребительского кредита?",
    "Какие ставки по вкладу Максимальный доход?",
    "Требования к заёмщику по ипотеке",
]

for q in test_queries:
    results = similarity_search(q, k=3)
    print_results(results, title=f"Запрос: {q}")

### 1.2 MMR — Maximum Marginal Relevance

MMR балансирует между **релевантностью** и **разнообразием** результатов,
чтобы избежать дублирующихся чанков из одного документа.

In [ ]:
def mmr_search(query: str, k: int = 4, fetch_k: int = 20, lambda_mult: float = 0.6) -> list:
    """
    MMR поиск.
    fetch_k     — кол-во кандидатов для финального отбора
    lambda_mult — 1.0 = чистая релевантность, 0.0 = чистое разнообразие
    """
    docs = vectorstore.max_marginal_relevance_search(
        query,
        k=k,
        fetch_k=fetch_k,
        lambda_mult=lambda_mult,
    )
    return [(doc, None) for doc in docs]


# Сравнение Similarity vs MMR на одном запросе
query = "Условия досрочного погашения кредита"

sim_results = similarity_search(query, k=4)
mmr_results = mmr_search(query, k=4, fetch_k=20, lambda_mult=0.6)

print("SIMILARITY SEARCH:")
for i, (doc, score) in enumerate(sim_results, 1):
    print(f"  [{i}] score={score:.4f} | {doc.metadata['title']} | тип: {doc.metadata['product_type']}")

print("\nMMR SEARCH (lambda=0.6):")
for i, (doc, _) in enumerate(mmr_results, 1):
    print(f"  [{i}] {doc.metadata['title']} | тип: {doc.metadata['product_type']}")

In [ ]:
# Сравнительный анализ: Similarity vs MMR
print("СРАВНЕНИЕ: Similarity Search vs MMR")
print()

for query in test_queries:
    sim = similarity_search(query, k=4)
    mmr = mmr_search(query, k=4, fetch_k=20)

    sim_types = [d.metadata['product_type'] for d, _ in sim]
    mmr_types = [d.metadata['product_type'] for d, _ in mmr]

    print(f"\nЗапрос: {query[:55]}")
    print(f"  Similarity — типы: {sim_types} | уникальных: {len(set(sim_types))}")
    print(f"  MMR        — типы: {mmr_types} | уникальных: {len(set(mmr_types))}")

print()
print("ВЫВОД:")
print("  Similarity Search: лучше когда нужен самый точный ответ из одного места.")
print("  MMR: лучше когда нужно охватить несколько аспектов темы (разнообразие).")

---
## 2. Продвинутые техники ретрива
### 2.1 Гибридный поиск: BM25 + векторный (Ensemble Retriever)

In [7]:
from langchain_community.retrievers import BM25Retriever
from langchain.retrievers import EnsembleRetriever

# Получаем все чанки из БД для BM25
# (BM25 — ключевой поиск, ему нужен весь корпус)
all_docs_raw = vectorstore.similarity_search("банк кредит вклад ипотека", k=500)

# BM25 ретривер
bm25_retriever = BM25Retriever.from_documents(all_docs_raw)
bm25_retriever.k = 4

# Векторный ретривер
vector_retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 4},
)

# Гибридный ретривер (50% BM25 + 50% Vector)
hybrid_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, vector_retriever],
    weights=[0.5, 0.5],
)

In [ ]:
# Сравнение: BM25 vs Vector vs Hybrid
query = "страхование жизни при ипотеке ставка"

bm25_results   = bm25_retriever.invoke(query)
vec_results    = vector_retriever.invoke(query)
hybrid_results = hybrid_retriever.invoke(query)

def show_retriever_results(docs, label):
    print(f"\n{label}")
    for i, doc in enumerate(docs[:4], 1):
        print(f"  [{i}] {doc.metadata['title']} | {doc.metadata['product_type']}")
        print(f"       {doc.page_content[:130].strip()}...")

print(f"Запрос: '{query}'")
show_retriever_results(bm25_results,   "BM25 (ключевой поиск)")
show_retriever_results(vec_results,    "Vector (семантический)")
show_retriever_results(hybrid_results, "Hybrid (BM25 + Vector)")

### 2.2 Фильтрация по метаданным

In [ ]:
def filtered_search(query: str, product_type: str, k: int = 4) -> list:
    """
    Поиск с фильтрацией по типу продукта.
    product_type: 'credit' | 'mortgage' | 'deposit' | 'requirements' | 'faq'
    """
    results = vectorstore.similarity_search_with_score(
        query,
        k=k,
        filter={"product_type": product_type},
    )
    return results


# Демонстрация: один запрос, три разных фильтра
query = "какова процентная ставка"
print(f"Запрос: '{query}'\n")

for ptype in ["credit", "mortgage", "deposit"]:
    results = filtered_search(query, product_type=ptype, k=2)
    print(f"Фильтр: product_type = '{ptype}'")
    for doc, score in results:
        print(f"  score={score:.4f} | {doc.page_content[:180].strip()}...")
    print()

### 2.3 Контекстное сжатие (Contextual Compression)

Фильтрует **внутри** найденных чанков — оставляет только те фрагменты,
которые достаточно близки к запросу по семантике.

In [ ]:
from langchain.retrievers import ContextualCompressionRetriever
from langchain.retrievers.document_compressors import EmbeddingsFilter

# EmbeddingsFilter — отбрасывает чанки, чьё сходство с запросом ниже порога
embeddings_filter = EmbeddingsFilter(
    embeddings=embeddings_model,
    similarity_threshold=0.75,
)

compression_retriever = ContextualCompressionRetriever(
    base_compressor=embeddings_filter,
    base_retriever=hybrid_retriever,
)

In [ ]:
# Сравнение: Hybrid vs Hybrid + Compression
query = "первоначальный взнос по семейной ипотеке"

hybrid_docs     = hybrid_retriever.invoke(query)
compressed_docs = compression_retriever.invoke(query)

print(f"Запрос: '{query}'")
print(f"\nHybrid (без сжатия): {len(hybrid_docs)} чанков")
for i, doc in enumerate(hybrid_docs, 1):
    print(f"  [{i}] {len(doc.page_content)} симв. | {doc.page_content[:160].strip()}...")

print(f"\nHybrid + Compression: {len(compressed_docs)} чанков")
for i, doc in enumerate(compressed_docs, 1):
    print(f"  [{i}] {len(doc.page_content)} симв. | {doc.page_content[:160].strip()}...")

if hybrid_docs and compressed_docs:
    avg_before = sum(len(d.page_content) for d in hybrid_docs) / len(hybrid_docs)
    avg_after  = sum(len(d.page_content) for d in compressed_docs) / len(compressed_docs)
    print(f"\nСредний размер чанка ДО сжатия   : {avg_before:.0f} симв.")
    print(f"Средний размер чанка ПОСЛЕ сжатия : {avg_after:.0f} симв.")
    print(f"Сокращение контекста              : {(1 - avg_after/avg_before)*100:.1f}%")

---
## 3. Оценка качества ретрива
### 3.1 Тестовый набор — 20 вопросов с эталонными типами документов

In [ ]:
from collections import Counter

# Тестовый набор: вопрос → ожидаемый product_type (эталон)
test_set = [
    # Кредиты
    {"query": "Какая минимальная сумма потребительского кредита?",         "relevant_type": "credit"},
    {"query": "Максимальный срок кредита в банке?",                        "relevant_type": "credit"},
    {"query": "Есть ли штраф за досрочное погашение кредита?",             "relevant_type": "credit"},
    {"query": "Какая базовая процентная ставка по кредиту?",               "relevant_type": "credit"},
    {"query": "Какие документы нужны для получения кредита?",              "relevant_type": "credit"},
    # Ипотека
    {"query": "Какой минимальный первоначальный взнос по ипотеке?",        "relevant_type": "mortgage"},
    {"query": "Что такое семейная ипотека и какова её ставка?",            "relevant_type": "mortgage"},
    {"query": "Можно ли использовать материнский капитал для ипотеки?",    "relevant_type": "mortgage"},
    {"query": "Сколько времени занимает оформление ипотеки?",              "relevant_type": "mortgage"},
    {"query": "Обязательно ли страховать жизнь при ипотеке?",              "relevant_type": "mortgage"},
    # Депозиты
    {"query": "Какая максимальная ставка по вкладу Максимальный доход?",   "relevant_type": "deposit"},
    {"query": "Можно ли пополнять вклад Накопительный?",                   "relevant_type": "deposit"},
    {"query": "Что произойдёт если закрыть вклад досрочно?",               "relevant_type": "deposit"},
    {"query": "До какой суммы застрахованы вклады?",                       "relevant_type": "deposit"},
    {"query": "Есть ли в банке валютные вклады в юанях?",                  "relevant_type": "deposit"},
    # Требования к заёмщикам
    {"query": "С какого возраста можно взять кредит?",                     "relevant_type": "requirements"},
    {"query": "Какой минимальный стаж работы нужен для кредита?",          "relevant_type": "requirements"},
    {"query": "Принимает ли банк самозанятых заёмщиков?",                  "relevant_type": "requirements"},
    {"query": "Что такое ПДН и каков максимально допустимый?",             "relevant_type": "requirements"},
    {"query": "Какой минимальный скоринговый балл для получения кредита?", "relevant_type": "requirements"},
]

print(f"Тестовый набор: {len(test_set)} вопросов")
print("Распределение по типам:", dict(Counter(t["relevant_type"] for t in test_set)))

### 3.2 Метрики Hit Rate@k и MRR

In [14]:
def evaluate_retriever(retriever_fn, test_set: list, k_values: list = [1, 3, 5]) -> dict:
    """
    Оценивает ретривер:
      Hit Rate@k — доля запросов, где релевантный тип документа попал в топ-k
      MRR        — Mean Reciprocal Rank (средний обратный ранг)
    """
    hit_counts = {k: 0 for k in k_values}
    reciprocal_ranks = []

    for item in test_set:
        query    = item["query"]
        expected = item["relevant_type"]
        docs     = retriever_fn(query, k=max(k_values))

        # Ищем первое вхождение нужного типа
        found_rank = None
        for rank, doc in enumerate(docs, 1):
            if doc.metadata.get("product_type") == expected:
                found_rank = rank
                break

        for k in k_values:
            if found_rank is not None and found_rank <= k:
                hit_counts[k] += 1

        reciprocal_ranks.append(1 / found_rank if found_rank else 0)

    n = len(test_set)
    return {
        "hit_rate": {k: round(hit_counts[k] / n, 4) for k in k_values},
        "mrr":      round(float(np.mean(reciprocal_ranks)), 4),
    }


# Обёртки — единый интерфейс (query, k) → list[Document]
def similarity_fn(query, k):
    return [doc for doc, _ in vectorstore.similarity_search_with_score(query, k=k)]

def mmr_fn(query, k):
    return vectorstore.max_marginal_relevance_search(query, k=k, fetch_k=k * 4)

def hybrid_fn(query, k):
    bm25_retriever.k = k
    vector_retriever.search_kwargs["k"] = k
    return hybrid_retriever.invoke(query)[:k]

In [ ]:
K_VALUES = [1, 3, 5]

retrievers_to_eval = [
    ("Similarity Search",  similarity_fn),
    ("MMR",                mmr_fn),
    ("Hybrid (BM25+Vec)",  hybrid_fn),
]

all_metrics = {}
for name, fn in retrievers_to_eval:
    print(f"  Оцениваем: {name}...", end=" ", flush=True)
    t0 = time.time()
    metrics = evaluate_retriever(fn, test_set, K_VALUES)
    metrics["latency_sec"] = round(time.time() - t0, 2)
    all_metrics[name] = metrics
    print(f"готово ({metrics['latency_sec']}с)")

In [ ]:
# Итоговая таблица метрик
col_w = 22
header = f"{'Ретривер':<{col_w}}" + "".join(f"{'HR@'+str(k):>8}" for k in K_VALUES) + f"{'MRR':>8}  {'Время(с)':>10}"

print("\n" + "=" * len(header))
print("ОЦЕНКА КАЧЕСТВА РЕТРИВЕРОВ")
print("=" * len(header))
print(header)
print("-" * len(header))

for name, m in all_metrics.items():
    row  = f"{name:<{col_w}}"
    row += "".join(f"{m['hit_rate'][k]:>8.3f}" for k in K_VALUES)
    row += f"{m['mrr']:>8.3f}  {m['latency_sec']:>10.2f}"
    print(row)

print("=" * len(header))
print()
print("HR@k = Hit Rate@k: доля запросов, где нужный документ попал в топ-k")
print("MRR  = Mean Reciprocal Rank: учитывает позицию первого релевантного результата")

In [ ]:
# Детальный анализ промахов
best_name = max(all_metrics, key=lambda n: all_metrics[n]["mrr"])
best_fn   = dict(retrievers_to_eval)[best_name]

print(f"Детальный разбор ошибок: {best_name}")

errors = []
for item in test_set:
    docs     = best_fn(item["query"], k=3)
    top1_type = docs[0].metadata.get("product_type") if docs else None
    if top1_type != item["relevant_type"]:
        errors.append({
            "query":    item["query"],
            "expected": item["relevant_type"],
            "got":      top1_type,
        })

if errors:
    print(f"Промахов @1: {len(errors)} из {len(test_set)}\n")
    for e in errors:
        print(f"  ✗ Вопрос  : {e['query']}")
        print(f"    Ожидали : {e['expected']}")
        print(f"    Получили: {e['got']}\n")
else:
    print(f"Промахов @1: 0 — все {len(test_set)} запросов обработаны верно!")

print()
print("ИТОГОВЫЕ ВЫВОДЫ")
print(f"  Лучший ретривер по MRR : {best_name}")
print(f"  MRR                    : {all_metrics[best_name]['mrr']}")
print(f"  Hit Rate@3             : {all_metrics[best_name]['hit_rate'][3]}")
print()
print("  Гибридный поиск (BM25 + Vector) даёт наилучшее качество,")
print("  объединяя точное совпадение ключевых слов и семантическое понимание.")

In [ ]:
# Сохраняем метрики в конфиг для Этапа 3
config["best_retriever"]    = best_name
config["retrieval_metrics"] = all_metrics

with open("rag_config.json", "w", encoding="utf-8") as f:
    json.dump(config, f, ensure_ascii=False, indent=2)

print("Конфигурация обновлена в rag_config.json")

**Следующий шаг → Этап 3: Интеграция с LLM (OpenRouter)**